# Analyse de données — API Deezer

Vous trouverez dans ce notebook les différentes parties attendues pour le test technique ainsi que mon argumentaire technique les accompagnant.
Le code de la partie ingestion est réalisé dans un load.py séparé que vous trouverez dans le repository.

## O. Préparation de l'environnement 
Ce projet nécessite uv installé sur la machine.
Vous trouverez les instructions à l'adresse [https://docs.astral.sh/uv/](https://docs.astral.sh/uv/)


In [1]:
!uv sync

Resolved 147 packages in 3ms
Checked 144 packages in 3ms


## Partie 1 : Récupération des données via l’API Deezer

Ma stratégie de collecte m'a amené à essayer plusieurs méthodes que je n'ai pas retenues.

La plus naïve était d'énumérer les ID de tracks Deezer en espérant qu'ils soient continus en fonction de la date de sortie du titre et qu'une majorité retourne un titre. De façon assez prévisible Deezer se prémunit contre cette énumération en ne peuplant pas tous leurs ID. Les quelques titres retournés sont des titres avec très peu d'écoutes faussant l'analyse pour la partie 2.

Ma deuxième approche était d'utiliser les playlists éditoriales par genre que Deezer met à disposition. Néanmoins cette approche fausse aussi toute étude statistique future car Deezer donne les tracks les plus populaires à l'instant T pour tout genre donné rendant plus difficile toute comparaison de popularité entre les genres.

L'approche que j'ai retenue est d'utiliser la programmation musicale des différents festivals avec lesquels Delight Data est partenaire.
En récupérant [cette liste de clients](https://www.delight-data.com/nos-clients) j'ai pu récupérer une programmation diverse et récente sur les dernières années. En prenant des festivals avec des programmations aussi diverses que Nancy Jazz Pulsations et Rock en Seine, j'ai pu récupérer une programmation éclectique et répondant aux contraintes d'équilibrage.


In [2]:
! uv run python load.py

Les Ardentes              7 playlists
Garorock                  9 playlists
Musilac                   4 playlists
Francofolies              5 playlists
We Love Green             2 playlists
Rock en Seine             3 playlists
Nancy Jazz Pulsations     4 playlists
Printemps de Bourges      1 playlists

4151 rows | 3128 tracks, 2712 albums, 1284 artists
  album 500/2712
  album 1000/2712
  album 1500/2712
  album 2000/2712
  album 2500/2712
albums    2645/2712
  track 500/3128
  track 1000/3128
  track 1500/3128
  track 2000/3128
  track 2500/3128
  track 3000/3128
tracks    3128/3128
  artist 500/1284
  artist 1000/1284
artists   1284/1284

done in 13.3 min


## Partie 2 : Analyse des données


Le script précédent stocke toutes les réponses de l'API dans une zone RAW stockée sur disque. 
Il est possible d'analyser ces fichiers JSONL avec DuckDB via du code SQL.

Cette approche est privilégiée vis-à-vis d'un script python/pandas par exemple principalement pour les performances de l'outil choisi ainsi que la facilité de lecture du SQL par tous. 

Le code pourrait ainsi facilement être adapté pour être intégré dans une pipeline dbt par exemple.

### 2.0 Préparation et analyse des données
DuckDB gère parfaitement l'import de json via la commande read_json_auto

In [2]:
import duckdb

con = duckdb.connect()
for entity in ["playlists", "playlist_tracks", "albums", "tracks", "artists"]:
    con.sql(f"""create view {entity} as
                select * from read_json_auto('data/raw/{entity}/data.jsonl',
                                             sample_size=-1)""")

con.sql("""
    select 'playlists officielles' as objet, count(*) as n from playlists
    union all select 'lignes de playlist', count(*) from playlist_tracks
    union all select 'titres uniques',     count(*) from tracks 
    union all select 'albums',             count(*) from albums
    union all select 'artistes',           count(*) from artists
""")

┌───────────────────────┬───────┐
│         objet         │   n   │
│        varchar        │ int64 │
├───────────────────────┼───────┤
│ playlists officielles │    35 │
│ lignes de playlist    │  4151 │
│ titres uniques        │  3128 │
│ albums                │  2645 │
│ artistes              │  1284 │
└───────────────────────┴───────┘

Partons du principe que nous voulons filtrer sur les titres sortis dans les années 2022-2025.
Mon hypothèse de base est que les titres sont assez bien répartis entre ces années, les artistes en festival aimant chanter leurs dernières sorties de l'année.

In [3]:
# Annees de sortie des TITRES (et non des editions de festival).
con.sql("""
    select year(release_date) as annee, count(*)
    from tracks
    where year(release_date) between 2022 and 2025
    group by 1 order by 1
""")

┌───────┬──────────────┐
│ annee │ count_star() │
│ int64 │    int64     │
├───────┼──────────────┤
│  2022 │          388 │
│  2023 │          434 │
│  2024 │          451 │
│  2025 │          463 │
└───────┴──────────────┘

In [4]:
con.sql("""
    create or replace view setlist_all as
    select pt._festival     as festival,
           pt._edition      as edition,
           t.id             as track_id,
           t.title          as titre,
           t.rank           as rank,
           t.duration       as duree_s,
           nullif(t.bpm, 0) as bpm,
           t.release_date   as sortie,
           ar.id            as artist_id,
           ar.name          as artiste,
           ar.nb_fan        as artiste_fans,
           al.id            as album_id,
           al.fans          as album_fans
    from playlist_tracks pt
    join tracks t         on t.id  = pt.id
    left join artists ar  on ar.id = t.artist.id
    left join albums  al  on al.id = t.album.id
""")

con.sql("""
    create or replace view setlist as
    select * from setlist_all where year(sortie) between 2022 and 2025
""")

# Un album porte zero, un ou plusieurs genres : une ligne par couple (titre, genre).
con.sql("""
    create or replace view setlist_genre as
    select s.*, g.name as genre
    from setlist s
    join albums al on al.id = s.album_id, unnest(al.genres.data) as u(g)
""")

con.sql("""
    create or replace view setlist_genre_all as
    select s.*, g.name as genre
    from setlist_all s
    join albums al on al.id = s.album_id, unnest(al.genres.data) as u(g)
""")


con.sql("""
    select count(distinct track_id)  as titres,
           count(distinct artist_id) as artistes,
           count(distinct album_id)  as albums,
           count(distinct festival)  as festivals
    from setlist
""")

┌────────┬──────────┬────────┬───────────┐
│ titres │ artistes │ albums │ festivals │
│ int64  │  int64   │ int64  │   int64   │
├────────┼──────────┼────────┼───────────┤
│   1736 │      870 │   1373 │         8 │
└────────┴──────────┴────────┴───────────┘

### 2.1 — le nombre de fans prédit-il la popularité des titres ?

On a dans les données `artiste.nb_fan` (nombre de fans de l'artiste), `album.fans` (nombre de fans de l'album) et `titre.rank` pour le rank du titre. 
On peut utiliser la fonction sql corrélation  **rho Spearman** afin de définir un facteur de corrélation entre ces différentes distributions.

In [5]:
con.sql("""
    with titres as (   
        select distinct track_id, rank, artiste_fans, album_fans from setlist
    ),


    rangs as (
        select rank() over (order by rank)         as r_rank,
               rank() over (order by artiste_fans) as r_fans,
               rank() over (order by album_fans)   as r_album
        from titres where artiste_fans is not null
    )


    select 'rank titre / fans artiste' as relation, count(*) as n,
           round(corr(r_rank, r_fans), 3) as rho_spearman from rangs
    
    union all

    select 'rank titre / fans album', count(*), round(corr(r_rank, r_album), 3) from rangs
""")

┌───────────────────────────┬───────┬──────────────┐
│         relation          │   n   │ rho_spearman │
│          varchar          │ int64 │    double    │
├───────────────────────────┼───────┼──────────────┤
│ rank titre / fans artiste │  1736 │        0.603 │
│ rank titre / fans album   │  1736 │        0.304 │
└───────────────────────────┴───────┴──────────────┘

On a une corrélation assez forte entre le rank du titre et le nombre de fans de l'artiste (0.6)
J'ai aussi calculé la corrélation entre le rank du titre et le nombre de fans de l'album qui rest positive mais est beaucoup moins nette.

### 2.2 — quels genres dominent, et leur durée ou BPM diffèrent-ils ?

On a uniquement une indication du genre à la maille album (d'ou la setlist spécifique définié avant cela)

In [6]:
con.sql("""
    select genre,
           count(distinct track_id)  as titres,
           count(distinct artist_id) as artistes,
           count(distinct festival)  as festivals,
           round(100.0 * count(distinct track_id)
                 / (select count(distinct track_id) from setlist), 1) as pct_corpus
    from setlist_genre
    group by genre
    having count(distinct track_id) >= 25
    order by titres desc
""")

┌───────────────────┬────────┬──────────┬───────────┬────────────┐
│       genre       │ titres │ artistes │ festivals │ pct_corpus │
│      varchar      │ int64  │  int64   │   int64   │   double   │
├───────────────────┼────────┼──────────┼───────────┼────────────┤
│ Rap/Hip Hop       │    531 │      219 │         8 │       30.6 │
│ Pop               │    245 │      133 │         8 │       14.1 │
│ Alternative       │    237 │      142 │         8 │       13.7 │
│ Electro           │    204 │      126 │         8 │       11.8 │
│ Dance             │    173 │      110 │         8 │       10.0 │
│ Rock              │    113 │       69 │         8 │        6.5 │
│ Chanson française │     91 │       42 │         8 │        5.2 │
│ Rap français      │     61 │       36 │         7 │        3.5 │
│ Rock indé         │     56 │       31 │         7 │        3.2 │
│ Jazz              │     53 │       40 │         6 │        3.1 │
│ R&B               │     45 │       35 │         8 │        2

Il y a clairement une domination du rap/hip hop dans les festivals et donc les playlists associées sur la période 2022-2025.
Viennent ensuite la pop, l'alternative et l'electro.
Notons que la majorité des festivals étudiés sont ecclectiques et ont donc chacun tous les genres les plus populaires.


---
Pour ce qui est du BPM, mon analyse indique qu'il est renseigné sur Deezer uniquement sur les titres anciens (cela ne doit pas faire partie des métadonnées fournies par le label).
Je vais porter mon analyse sur les quelques titres anciens comportant cette métrique

In [7]:
con.sql("""
    select case when year(sortie) between 2022 and 2025 then '2022-2025'
                else 'ancien' end as perimetre,

           count(distinct track_id) as titres,
           count(distinct track_id) filter (bpm is not null) as titres_avec_bpm,
           round(100.0 * count(distinct track_id) filter (bpm is not null)
                 / count(distinct track_id), 1) as couverture_bpm_pct
                 
    from setlist_all group by 1
""")

┌───────────┬────────┬─────────────────┬────────────────────┐
│ perimetre │ titres │ titres_avec_bpm │ couverture_bpm_pct │
│  varchar  │ int64  │      int64      │       double       │
├───────────┼────────┼─────────────────┼────────────────────┤
│ ancien    │   1392 │             655 │               47.1 │
│ 2022-2025 │   1736 │               3 │                0.2 │
└───────────┴────────┴─────────────────┴────────────────────┘

---

In [12]:
con.sql("""
    select genre,
           count(*)              as titres_avec_bpm,
           round(avg(bpm), 1)    as bpm_moyen,
           round(median(bpm), 1) as bpm_median,
           round(stddev(bpm), 1) as ecart_type,
    from (select distinct track_id, genre, bpm from setlist_genre_all where bpm is not null)
    group by genre having count(*) >= 15
    order by bpm_moyen desc
""")



┌───────────────────┬─────────────────┬───────────┬────────────┬────────────┐
│       genre       │ titres_avec_bpm │ bpm_moyen │ bpm_median │ ecart_type │
│      varchar      │      int64      │  double   │   double   │   double   │
├───────────────────┼─────────────────┼───────────┼────────────┼────────────┤
│ Rap/Hip Hop       │             197 │     131.6 │      132.1 │       23.5 │
│ Alternative       │              64 │     130.7 │      126.8 │       25.3 │
│ Rap français      │              22 │     128.7 │      126.8 │       21.5 │
│ Pop               │             116 │     126.7 │      122.5 │       25.0 │
│ Jazz              │              15 │     123.6 │      121.6 │       25.0 │
│ Rock              │              63 │     121.6 │      116.8 │       22.5 │
│ Chanson française │              16 │     120.7 │      119.5 │       20.8 │
│ Electro           │              63 │     119.1 │      118.1 │       17.8 │
│ Dance             │              41 │     118.4 │      119.8 │

De facon surprenante, tous les genres ont un BPM moyen assez stable (entre 120 et 130 bpm). Il n'y a pas de différence évidente.

---

In [9]:
con.sql("""
    select genre,
           count(*)                   as titres,
           round(avg(duree_s))        as duree_moy_s,
           round(median(duree_s))     as duree_med_s,
           round(1.96 * stddev(duree_s) / sqrt(count(*))) as ic95_s
    from (select distinct track_id, genre, duree_s from setlist_genre)
    group by genre having count(*) >= 25
    order by duree_moy_s desc
""")

┌───────────────────┬────────┬─────────────┬─────────────┬────────┐
│       genre       │ titres │ duree_moy_s │ duree_med_s │ ic95_s │
│      varchar      │ int64  │   double    │   double    │ double │
├───────────────────┼────────┼─────────────┼─────────────┼────────┤
│ Jazz              │     53 │       254.0 │       239.0 │   25.0 │
│ Electro           │    204 │       236.0 │       223.0 │   10.0 │
│ Rock              │    113 │       230.0 │       215.0 │   13.0 │
│ Dance             │    173 │       224.0 │       214.0 │   10.0 │
│ Alternative       │    237 │       213.0 │       203.0 │    8.0 │
│ Techno/House      │     27 │       211.0 │       189.0 │   28.0 │
│ Rock indé         │     56 │       205.0 │       198.0 │   13.0 │
│ Pop Indé          │     25 │       196.0 │       199.0 │    8.0 │
│ Pop               │    245 │       194.0 │       189.0 │    5.0 │
│ Chanson française │     91 │       192.0 │       187.0 │    7.0 │
│ R&B               │     45 │       192.0 │    

Enfin pour ce qui est de l'analyse de la longueur des tracks, on retrouve une echelle qui va du jazz tout en longueur au rap francaus qui est beaucoup plus court


### 2.4 Question 3 — les affiches de festivals se ressemblent-elles ?

**Hypothèse.** Je pense que même si les festivals ont été choisis pour leur complémentarité, on retrouvera forcément un groupe d'artiste faisant la tournée de tous les festivals.
De meme je pense que certains festivals vont avoir plus de facilités à faire venir de grosses têtes d'affiches avec un rank plus haut que des petits festivals comme Nancy Jazz


In [11]:
con.sql("""
    with par_artiste as (   -- un artiste compte une fois par festival, pas une fois par titre
        select festival, artist_id, any_value(artiste_fans) as fans
        from setlist where artist_id is not null
        group by festival, artist_id
    )


    select festival,
           count(*)                        as artistes,
           round(median(fans))             as fans_median,
           max(fans)                       as fans_du_plus_gros_artiste,
        
           round(100.0 * max(fans) / sum(fans), 1) as pct_fans_du_1er
    from par_artiste group by festival order by fans_median desc
""")

┌───────────────────────┬──────────┬─────────────┬───────────────────────────┬─────────────────┐
│       festival        │ artistes │ fans_median │ fans_du_plus_gros_artiste │ pct_fans_du_1er │
│        varchar        │  int64   │   double    │           int64           │     double      │
├───────────────────────┼──────────┼─────────────┼───────────────────────────┼─────────────────┤
│ Les Ardentes          │      228 │     84903.0 │                  24082041 │            13.5 │
│ Francofolies          │      107 │     55356.0 │                   8944739 │            14.1 │
│ Garorock              │      181 │     36920.0 │                  20106944 │            17.9 │
│ Musilac               │      115 │     36351.0 │                   9783213 │            23.5 │
│ We Love Green         │      112 │     23248.0 │                  18385936 │            28.1 │
│ Printemps de Bourges  │       72 │     14814.0 │                   6891325 │            45.9 │
│ Rock en Seine         │     

Les ardentes sont le festival avec le nombre de fans médian des artistes le plus élevé tandis que nancy jazz est en derniere position

La colonne pct_fans_du_1er donne la structure de l'affiche. Au Printemps de Bourges, GIMS pèse à lui seul 46 % de l'audience cumulée : il est une tête d'affiche vraiment ecrasante si on fait la somme des fans
À l'inverse Nancy Jazz (11 %) et Rock en Seine (12,6 %) ont des affiches homogènes, sans nom écrasant 

---

In [35]:
con.sql("""
    select nb_festivals, count(*) as artistes
    from (select artist_id, count(distinct festival) as nb_festivals
          from setlist where artist_id is not null
          group by artist_id)
    group by nb_festivals
    order by artistes desc
""")

┌──────────────┬──────────┐
│ nb_festivals │ artistes │
│    int64     │  int64   │
├──────────────┼──────────┤
│            1 │      704 │
│            2 │      127 │
│            3 │       30 │
│            4 │        8 │
│            5 │        1 │
└──────────────┴──────────┘

In [37]:
con.sql("""
    select ar.artiste,
           count(distinct s.festival) as nb_festivals,
           string_agg(distinct s.festival, ', ' order by s.festival) as affiches,
           any_value(ar.fans) as fans
    from (select artist_id, any_value(artiste) as artiste, any_value(artiste_fans) as fans
          from setlist where artist_id is not null group by artist_id) ar
    join setlist s using (artist_id)
    group by ar.artiste
    having count(distinct s.festival) >= 4
    order by nb_festivals desc, fans desc
""")

┌──────────────────┬──────────────┬───────────────────────────────────────────────────────────────────────────┬─────────┐
│     artiste      │ nb_festivals │                                 affiches                                  │  fans   │
│     varchar      │    int64     │                                  varchar                                  │  int64  │
├──────────────────┼──────────────┼───────────────────────────────────────────────────────────────────────────┼─────────┤
│ miki             │            5 │ Francofolies, Les Ardentes, Musilac, Nancy Jazz Pulsations, Rock en Seine │   51515 │
│ GIMS             │            4 │ Francofolies, Garorock, Les Ardentes, Printemps de Bourges                │ 6891325 │
│ disiz            │            4 │ Garorock, Les Ardentes, Nancy Jazz Pulsations, We Love Green              │  449367 │
│ Charlotte Cardin │            4 │ Garorock, Musilac, Printemps de Bourges, We Love Green                    │  281003 │
│ Feu! Chatterton  │    

Enfin j'avais tort sur mon premier point l'immense majorité des chanteurs ne font qu'un seul festival, et la répétition des têtes d'affiche est plus une exception que la règle.

Vous en retrouverez la liste ci-dessus.
